# Advanced Retrieval with the Cat Health PDF

Session 1 used a dense vector retriever:

```text
question -> embed -> nearest chunks -> answer
```

This notebook keeps the same cat-health PDF and compares dense vector search, BM25, parent-child retrieval, hybrid retrieval, second-stage reranking, and multi-query retrieval.

> This is a retrieval exercise, not veterinary guidance. Answer only from retrieved context. Recommend a veterinarian for diagnosis, treatment, medication, or urgent-care decisions.

> **🛠️ Firewall note — Cohere → LLM reranker.** The original course material used **Cohere Rerank** for the second stage. The corporate firewall blocks Cohere, so this notebook substitutes an **OpenAI LLM scorer** in `rerank_parent_candidates` (Task 7). The function signature, return shape, and downstream pipelines are identical, so every retriever, evaluator, and answer cell keeps working unchanged. See *Appendix A: Cohere Rerank firewall workaround* in `SESSION_NOTES.md` for the full rationale and per-cell diff.


## Learning Outcomes

By the end of this session, you will be able to:

- Explain the different failure modes of dense and BM25 retrieval.
- Fuse independent ranked lists with reciprocal rank fusion (RRF).
- Increase recall with multiple generated search queries.
- Search focused child chunks while returning useful parent-page context.
- Use a **second-stage reranker** as a quality filter on top of hybrid retrieval. The course material introduces this with Cohere Rerank; this notebook implements it with an OpenAI LLM scorer because Cohere is firewalled, but the lesson — *retrieve broadly, then rerank* — is the same.
- Compare retrieval systems with reviewed cases, visible evidence, metrics, and latency.


## Build Order

Build and compare the following layers:

1. Naive dense RAG: in-memory Qdrant, OpenAI embeddings, nearest child chunks, and a grounded answer.
2. BM25: sparse lexical retrieval over the same chunks.
3. Parent-child retrieval: search precise child chunks and return their parent PDF pages.
4. Hybrid retrieval with second-stage reranking: fuse dense and BM25 candidates with reciprocal rank fusion (RRF), recover parent pages, then rerank them. *Originally Cohere Rerank; this notebook uses an OpenAI LLM scorer because Cohere is blocked by the corporate firewall (see Task 7).*
5. Multi-query retrieval: generate alternate searches before the hybrid retrieve-then-rerank pipeline.

Dense retrieval plus BM25 is **hybrid retrieval** (or hybrid search). RRF is an **ensemble** method that combines their ranked lists. Adding a second-stage reranker (Cohere Rerank in the original material, an LLM scorer here) creates a **two-stage hybrid retrieve-then-rerank pipeline**.

Extra stages add latency, cost, and sometimes noise. Evaluate each added stage against those trade-offs.


## Task 1: Setup

Install the project environment from the session folder with `uv sync`, then run this notebook from that same folder. You only need `OPENAI_API_KEY` available when the notebook prompts for it.

> The original course material also required `COHERE_API_KEY` for second-stage reranking. That requirement has been **removed** because Cohere's API is blocked by the corporate firewall. The reranker now reuses the same OpenAI credentials as the rest of the notebook (see Task 7 for the implementation and the `SESSION_NOTES.md` appendix for the rationale).


In [1]:
from __future__ import annotations

import os
import re
from dataclasses import replace
from getpass import getpass
from pathlib import Path
from typing import Iterable, Sequence

import pandas as pd
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from rank_bm25 import BM25Okapi

from lib import (
    AnswerOutput,
    EvalCase,
    EvalItem,
    RetrievedDocument,
    answer_similarity_scorer,
    compare_eval_reports,
    compare_reports,
    faithfulness_scorer,
    make_openai_faithfulness_judge,
    run_eval,
    run_retrieval_eval,
)


/var/folders/z8/pbvjy1gs1d3b_rd53b4knls80000gp/T/ipykernel_34164/2255838480.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

CHAT_MODEL = os.environ.get("AIM_CHAT_MODEL", "gpt-5.4-mini")
EVAL_MODEL = os.environ.get("AIM_EVAL_MODEL", CHAT_MODEL)
EMBEDDING_MODEL = os.environ.get("AIM_EMBEDDING_MODEL", "text-embedding-3-small")
# Cohere Rerank is unavailable (firewalled); we use an OpenAI LLM as a second-stage reranker instead.
RERANK_MODEL = os.environ.get("AIM_RERANK_MODEL", CHAT_MODEL)

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
answer_model = ChatOpenAI(model=CHAT_MODEL, temperature=0)
evaluation_model = ChatOpenAI(model=EVAL_MODEL, temperature=0)
query_model = ChatOpenAI(model=CHAT_MODEL, temperature=0)
rerank_model = ChatOpenAI(model=RERANK_MODEL, temperature=0)

print(f"Chat model: {CHAT_MODEL}")
print(f"Evaluation model: {EVAL_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LLM rerank model: {RERANK_MODEL}")


Chat model: gpt-5.4-mini
Evaluation model: gpt-5.4-mini
Embedding model: text-embedding-3-small
LLM rerank model: gpt-5.4-mini


## Task 2: Load and Chunk the Cat PDF for Naive RAG

Load the bundled PDF as page-level documents, then split each page into focused chunks for the dense RAG baseline. Preserve source and page metadata on every chunk.


In [3]:
pdf_path = Path("data/cat_health_guidelines.pdf")
if not pdf_path.exists():
    raise FileNotFoundError(f"Expected the cat PDF at {pdf_path.resolve()}")

pages = [page for page in PyPDFLoader(str(pdf_path)).load() if page.page_content.strip()]
for page_number, page in enumerate(pages, start=1):
    page.metadata.update(
        {
            "source": pdf_path.name,
            "page": page_number,
            "page_id": f"page-{page_number:02d}",
            "document_type": "cat_health_guideline",
        }
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")
print(pages[0].metadata)


Loaded 22 text-containing PDF pages.
{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 1, 'page_label': '1', 'page_id': 'page-01', 'document_type': 'cat_health_guideline'}


In [4]:
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    add_start_index=True,
)
children = child_splitter.split_documents(pages)
for index, child in enumerate(children):
    child.metadata["chunk_id"] = f"child-{index:03d}"

print(f"Created {len(children)} child chunks from {len(pages)} parent pages.")
print(children[0].metadata)
print(children[0].page_content[:500])


Created 159 child chunks from 22 parent pages.
{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 1, 'page_label': '1', 'page_id': 'page-01', 'document_type': 'cat_health_guideline', 'start_index': 0, 'chunk_id': 'child-000'}
VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are publishe


### ❓ Question 1: Traceability

Why should every child chunk retain its source file and page metadata?


### Question 1 Response: Traceability

Every child chunk should retain `source`, `page`, `page_id`, and `chunk_id` because retrieval is only the **first** stage of a RAG pipeline — the same chunks are then used for grounding, citation, evaluation, and debugging. Without per-chunk provenance you lose the ability to:

1. **Cite the answer back to the user.** The system prompt in Task 3 ends with "Sources line", and the answer model emits citations like `child-034 p.7`. Without `chunk_id` and `page`, the model can only paraphrase — the user cannot verify the claim.
2. **Recover the parent page** (Task 5). Parent-child retrieval keys off `page_id`. Without it, we cannot turn focused child chunks back into full-page context.
3. **Score retrieval honestly** (Task 9). `EvalCase.relevant_evidence_ids` references `page-03`, `page-06`, etc. The eval harness compares those against `evidence_ids` on each `RetrievedDocument`, which originate from `page_id` on the child. Strip the metadata and Hit@k / MRR collapse to zero — even when the retriever did the right thing.
4. **Filter by document type** when you add a second corpus. `document_type = "cat_health_guideline"` lets a future router avoid mixing evidence from unrelated PDFs (Advanced Builds section).
5. **Debug retrieval failures.** When the answer is wrong, the first question is always "what chunks did we retrieve?" Stable IDs and page numbers let a human open the PDF, read the page, and decide whether the bug is in chunking, retrieval, or generation.

In short: metadata is the thread that connects retrieval, generation, evaluation, and human review. Lose it and you have a chatbot, not a RAG system.


### Shared Result Representation

All retrievers return the same RetrievedDocument type. Stable chunk and page IDs make the evidence inspectable and keep later comparisons fair.


In [5]:
def as_retrieved_document(document: Document, score: float | None = None) -> RetrievedDocument:
    return RetrievedDocument(
        id=document.metadata["chunk_id"],
        text=document.page_content,
        score=float(score) if score is not None else None,
        evidence_ids=(document.metadata["page_id"],),
        metadata=dict(document.metadata),
    )


def print_results(results: Sequence[RetrievedDocument], text_limit: int = 260) -> None:
    for rank, result in enumerate(results, start=1):
        page = result.metadata.get("page", "?")
        score = "n/a" if result.score is None else f"{result.score:.4f}"
        print(f"#{rank} | {result.id} | page={page} | score={score}")
        print(result.text[:text_limit].replace("\n", " "))
        print()


## Task 3: Naive Dense Vector RAG with In-Memory Qdrant

Session 1 baseline:

question -> OpenAI embedding -> Qdrant nearest-neighbor search -> answer

Create an in-memory Qdrant collection from the child chunks. Retrieve the nearest chunks, then pass only those chunks to the answer model. Use this **naive RAG baseline** for later comparisons.


In [6]:
# Qdrant stays in memory for this notebook. It disappears when the kernel stops.
vector_store = QdrantVectorStore.from_documents(
    documents=children,
    embedding=embeddings,
    location=":memory:",
    collection_name="cat_health_naive_dense_rag",
)

FIRST_STAGE_K = 8


def dense_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    matches = vector_store.similarity_search_with_score(question, k=k)
    return [as_retrieved_document(document, score) for document, score in matches]


dense_preview = dense_retrieve("What should a senior cat wellness visit cover?", k=4)
print_results(dense_preview)


#1 | child-034 | page=7 | score=0.6789
disease. Mature Adult and Senior Cats The medical history and examination of mature adult and senior cats will be focused on early detection of disease. Adult and senior cats are often diagnosed with comorbidities. Speci ﬁc questions regarding changes in appet

#2 | child-021 | page=6 | score=0.6741
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care re

#3 | child-060 | page=10 | score=0.6188
study, three 10- to 15-minute exercise sessions per day led to a loss of approximately 1% of body weight in 1 month with no food intake restrictions. 66 Senior Cats Senior cats exhibiting new or unusual behavior should be evaluated for medical conditions. 12 C

#4 | child-036 | page=8 | score=0.6182
Detecting signs of pain or anxiety and evaluation of qual

### Naive RAG Answer

Retrieval quality and answer quality are separate concerns. Pass the nearest dense chunks to the answer model. The same grounded-answer function will later receive context from the other retrievers.


In [7]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer only from the provided cat-health guideline context. "
            "Do not diagnose, prescribe, or make an urgent-care decision. "
            "If the context is insufficient, say so. End with a short Sources line.",
        ),
        ("human", "Question: {question}\n\nContext:\n{context}"),
    ]
)


def format_context(documents: Sequence[RetrievedDocument]) -> str:
    blocks = []
    for document in documents:
        page = document.metadata.get("page", "unknown")
        blocks.append(f"[source={document.id}; page={page}]\n{document.text}")
    return "\n\n".join(blocks)


def answer_with_retriever(
    question: str,
    retriever,
    k: int = 4,
) -> AnswerOutput:
    documents = retriever(question, k=k)
    response = answer_model.invoke(
        answer_prompt.format_messages(question=question, context=format_context(documents))
    )
    return AnswerOutput(answer=str(response.content), documents=tuple(documents))


naive_result = answer_with_retriever(
    "What should an owner discuss at a senior cat wellness visit?",
    retriever=dense_retrieve,
)
print(naive_result.answer)
print("\nNaive dense evidence:")
print_results(naive_result.documents, text_limit=200)


At a senior cat wellness visit, an owner should discuss any changes in:

- appetite
- urination and drinking habits
- vomiting, hairball vomiting, or diarrhea
- nighttime activity or vocalization
- normal habits or activity level
- litter box use
- mobility, balance, pain, or vision changes

The visit is aimed at early detection of disease and identifying possible cognitive dysfunction, reduced mobility, pain, or reduced vision. If you want, I can also summarize what the guideline says about how often senior cats should be seen.  

Sources: child-034 p.7; child-060 p.10; child-021 p.6

Naive dense evidence:
#1 | child-034 | page=7 | score=0.6958
disease. Mature Adult and Senior Cats The medical history and examination of mature adult and senior cats will be focused on early detection of disease. Adult and senior cats are often diagnosed with 

#2 | child-021 | page=6 | score=0.6693
For example, some senior cats aged 10 years and older may remain in excellent physical condition and woul

## Task 4: BM25 Sparse Retrieval

BM25 ranks the same child chunks with lexical term matches rather than embeddings. It is useful for abbreviations, age ranges, named conditions, and phrases that dense similarity may blur.

Keep the corpus, chunks, and retrieval depth fixed. The retriever is the only thing that changes.


In [8]:
def tokenize(text: str) -> list[str]:
    return re.findall(r"[a-z0-9]+", text.lower())


bm25 = BM25Okapi([tokenize(child.page_content) for child in children])


def bm25_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    scores = bm25.get_scores(tokenize(question))
    ranked_indices = sorted(range(len(scores)), key=lambda index: scores[index], reverse=True)
    return [
        as_retrieved_document(children[index], float(scores[index]))
        for index in ranked_indices[:k]
    ]


bm25_preview = bm25_retrieve("What do BCS and MCS stand for?", k=4)
print_results(bm25_preview)


#1 | child-079 | page=13 | score=11.1440
could result in health problems. 83 No matter the life stage, to help avoid potential nutrient insufﬁciencies, cats should be fed diets labeled with an Association of American Feed Control Of ﬁcials statement of nutritional adequacy. AAHA and the AAFP do not a

#2 | child-029 | page=6 | score=10.8317
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical examination at all life stages to allow early 56 JAAHA | 57:2 Mar/Apr 2021

#3 | child-006 | page=1 | score=9.7786
BCS (body condition score); DER (daily energy requirements); DJD (degenerative joint disease); FCV (feline calicivirus); FeLV (feline leukemia virus); FHV-1 (feline herpesvirus type 1); FIC (feline idiopathic cystitis); FPV (feline panleukopenia virus); GI (ga

#4 | child-082 | page=13 | score=9.7514
vidual patient. Recommendations can be found in the AAFP ’s Feline Feeding Programs Consensus S

### ❓ Question 2: When Should BM25 Win?

Which query is more likely to favor BM25: `What do BCS and MCS stand for?` or `How should an owner get ready for a senior-cat wellness visit?` Why?


### Question 2 Response: When Should BM25 Win?

**BM25 will favor "What do BCS and MCS stand for?"**

Why:
- The query contains two **rare, high-IDF tokens** — `BCS` and `MCS` — that appear in only a handful of chunks. BM25's term-weighting promotes documents containing exactly those tokens.
- Dense embeddings, by contrast, treat acronyms as low-information surface forms. The semantic vector for "BCS" is close to many anatomy/measurement terms, so dense similarity *spreads* its score across many plausible chunks instead of concentrating it on the right one. The output above shows this: dense puts `child-005` (a generic disclaimer) at rank 1 with score 0.43, while BM25 puts the actual definition chunk `child-029` at rank 1 with score 12.3.

By contrast, "How should an owner get ready for a senior-cat wellness visit?" is **paraphrase-heavy**:
- The gold passage uses different surface words ("annual examination", "Discussion Items for All Life Stages", "appointment template"). None of "get ready", "prepare", "wellness", or "owner" appear verbatim in the gold chunk.
- BM25 has no way to bridge that lexical gap and is easily misled by chunks that happen to contain "cats" or "senior".
- Dense embeddings encode meaning, so synonyms and rephrasings still land near the gold chunk in vector space.

**Heuristic to remember:** if the user's query contains rare, content-bearing tokens that literally appear in the corpus (acronyms, drug names, named conditions, age ranges, units), prefer BM25. If the query is a paraphrase, a question, or uses generic words, prefer dense. Use **hybrid + RRF** when you can't tell which mode the query falls into.


### 🚀 Activity 1: Dense vs. BM25 Failure Modes

Pick three questions: one paraphrase-heavy question, one exact-term question, and one broad multi-part question. Inspect both result lists before generating an answer.

- Which retriever put the best evidence first?
- Which retriever returned redundant chunks?
- Which question type should favor BM25, and why?


In [9]:
activity_questions = [
    "What does the guideline say about BCS and MCS?",
    "How often should senior cats see a veterinarian?",
    "What factors shape an individualized preventive-care plan?",
]

for question in activity_questions:
    print(f"\nQUESTION: {question}\n")
    print("DENSE")
    print_results(dense_retrieve(question, k=3), text_limit=150)
    print("BM25")
    print_results(bm25_retrieve(question, k=3), text_limit=150)



QUESTION: What does the guideline say about BCS and MCS?

DENSE
#1 | child-005 | page=1 | score=0.4263
mendations should not be construed as dictating an exclusive protocol, course of treatment, or procedure. Variations in practice may be war- ranted ba

#2 | child-029 | page=6 | score=0.4119
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical exami

#3 | child-006 | page=1 | score=0.3845
BCS (body condition score); DER (daily energy requirements); DJD (degenerative joint disease); FCV (feline calicivirus); FeLV (feline leukemia virus);

BM25
#1 | child-029 | page=6 | score=12.3015
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical exami

#2 | child-082 | page=13 | score=11.5603
vidual patient. Recommendations can be found in the AAFP ’s Feline Feeding Programs Consensus Statement. 19 Young Adult Cats

### Activity 1 Reflection: Dense vs. BM25 Failure Modes

Inspecting the three question types from the output above:

| Question type | Question | Best retriever at rank 1 | Redundancy |
| --- | --- | --- | --- |
| Exact term | "What does the guideline say about BCS and MCS?" | **BM25** — child-029 (page 6, the actual BCS/MCS recording sentence) outranks dense (which puts a boilerplate disclaimer at rank 1) | Dense rank 1 is near-noise; BM25 is tightly on-topic |
| Paraphrase | "How often should senior cats see a veterinarian?" | **Dense** — child-021 (page 6, life-stage frequency) is rank 1 for both, but BM25's rank 1 is child-072 (page 12, *marking behaviour*) — an irrelevant lexical match on "cats" | BM25 returns a clearly wrong top result |
| Broad multi-part | "What factors shape an individualized preventive-care plan?" | **Tie** — both top child-023 (page 6 appointment template). Dense's rank 2-3 are more on-strategy; BM25 drifts to child-025/001 | BM25 has higher topical drift |

**Answers to the activity prompts:**

1. **Which retriever put the best evidence first?**
   - **Exact-term query:** BM25 — it scored the literal acronyms `BCS` and `MCS` heavily and placed the canonical recording sentence at rank 1. Dense buried that same chunk at rank 2 behind a generic disclaimer page (child-005) because semantic similarity treats abbreviations as low-information tokens.
   - **Paraphrase query:** Dense — "how often should senior cats see a veterinarian" has no exact lexical match for "frequency" or "examination cadence" in the gold chunk. BM25 was misled by the high-frequency word "cats" and surfaced a page on marking behaviour.
   - **Broad multi-part query:** Roughly a tie at rank 1; dense's tail (ranks 2-3) was more coherent.

2. **Which retriever returned redundant chunks?**
   - **Dense** returned multiple page-1 chunks (`child-005`, `child-006`) for the BCS/MCS query — both are from the same boilerplate front-matter page. Dense embedding similarity is happy to return near-duplicates because they live near each other in vector space.
   - **BM25** showed redundancy of a different kind: when a term appears in unrelated contexts, it returns multiple *topically irrelevant* chunks that happen to share the term ("cats" on the marking-behaviour page).
   - This is exactly why **RRF + reranking** in Task 7 helps — it pulls each retriever's strengths together and breaks ties using a second signal.

3. **Which question type should favor BM25, and why?**
   - **Exact-term queries** (acronyms, named conditions, age ranges, drug names). BM25's TF-IDF scoring rewards documents that contain those specific tokens with low document frequency, exactly the signal embeddings *flatten*. The BCS/MCS query is the canonical example: BM25 score 12.3 for the right chunk, dense similarity only 0.41.
   - **Paraphrase-heavy queries** should favor dense embeddings, because semantic similarity captures meaning even when the surface words differ.
   - **Broad multi-part queries** benefit most from the *combination* (hybrid + RRF), because neither retriever alone covers all sub-aspects.


## Task 5: Parent-Child Retrieval

Build a parent-child pipeline on top of the dense retriever:

question -> dense child search in Qdrant -> matching parent PDF pages

Child chunks give the vector store a focused search surface. Parent pages give the answer model surrounding context. Parent-child retrieval adds context recovery to dense retrieval; it is not hybrid retrieval.


In [10]:
# Parent-page lookup begins here; naive RAG does not need parent records.
parents_by_id = {page.metadata["page_id"]: page for page in pages}


def recover_parent_documents(
    child_candidates: Sequence[RetrievedDocument],
    k: int,
) -> list[RetrievedDocument]:
    parents: list[RetrievedDocument] = []
    seen_parent_ids: set[str] = set()

    for child in child_candidates:
        page_id = child.metadata["page_id"]
        if page_id in seen_parent_ids:
            continue
        parent = parents_by_id[page_id]
        parents.append(
            RetrievedDocument(
                id=page_id,
                text=parent.page_content,
                score=child.score,
                evidence_ids=(page_id,),
                metadata={
                    **parent.metadata,
                    "retrieved_from_child": child.id,
                    "first_stage_score": child.score,
                },
            )
        )
        seen_parent_ids.add(page_id)
        if len(parents) == k:
            break
    return parents


def parent_child_dense_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = dense_retrieve(question, k=FIRST_STAGE_K)
    return recover_parent_documents(child_candidates, k=k)


parent_preview = parent_child_dense_retrieve(
    "How should an owner prepare for a senior cat wellness visit?",
    k=3,
)
print_results(parent_preview, text_limit=400)


#1 | page-06 | page=6 | score=0.6861
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force recommends a minimum of annual examinations for all ca

#2 | page-07 | page=7 | score=0.6555
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner recognize subtle changes. Diseases and conditions that require additional focus during the examination by each life stage are listed in Table 3. Kittens Kittens will have different health risks dependin

#3 | page-10 | page=10 | score=0.6434
including carpeting, window and door frames, curtains, and couches. Keeping the nail

### ❓ Question 3: Search Small, Return Large?

Why does parent-child retrieval search focused child chunks but return the larger parent pages instead of indexing and returning only full pages?


### Question 3 Response: Search Small, Return Large?

Child chunks (~800 chars) and parent pages serve **two different roles**, and indexing only one of them gives up one of the two.

**Why search the small children:**
- Embedding similarity is computed over a *single dense vector per chunk*. A whole PDF page mixes many topics (e.g., page 6 covers life-stage definitions, examination cadence, *and* BCS/MCS recording). Averaging those topics into one embedding **dilutes** the signal for any specific query — the page's vector ends up "near" everything and "perfectly near" nothing.
- BM25 has the same problem: term-frequency saturation on a long page hides the most relevant sentence among unrelated terms.
- Small, focused chunks produce **sharper, more discriminating** vectors and BM25 scores. Hit-rate and MRR rise.

**Why return the large parents:**
- The answer model needs **enough surrounding context** to ground its answer. An 800-char child often cuts mid-sentence or mid-list, dropping the qualifier the user actually asked about ("for cats *over 10 years*…").
- Stable parent IDs (`page-01`, `page-02`, …) make citation, deduplication, and human review easy. If two children hit the same page, parent-child collapses them to one returned document.
- Pages are a natural **semantic unit** in this PDF — the AAHA guideline editors arranged content with page-level coherence.

**Why not just index the pages?** You'd lose retrieval quality (see "Why search the small children") to save context quality. Parent-child gives you both.

**Why not just return the children?** You'd lose context recovery and increase the chance of cutting an answer-critical sentence in half.

In short: **index the unit that maximizes retrieval precision; return the unit that maximizes answer grounding.** They don't have to be the same unit.


# Breakout Room #2: Combine, Rank, and Expand

The first half produced three building blocks: dense retrieval, BM25, and parent-child context recovery. This breakout combines them and compares the results.

In this breakout:

1. Fuse dense and BM25 rankings with reciprocal rank fusion.
2. Rerank the broad hybrid candidate set with Cohere.
3. Expand vague questions into multiple searches.
4. Compare the trade-offs with the local evaluation library.

Keep only the stages that improve measured retrieval enough to justify their latency.


## Task 6: Hybrid Retrieval — Dense + BM25

Dense and BM25 scores use different scales, so raw scores cannot be added directly. Reciprocal rank fusion (RRF) works from their **ranked lists** instead.

The hybrid first stage is:

dense candidates + BM25 candidates -> RRF -> hybrid child candidates

**Hybrid retrieval** combines semantic vector retrieval with sparse lexical retrieval. RRF makes it an **ensemble retriever** by fusing multiple rankings.


In [11]:
def reciprocal_rank_fusion(
    ranked_lists: Iterable[Sequence[RetrievedDocument]],
    *,
    limit: int,
    rrf_constant: int = 60,
) -> list[RetrievedDocument]:
    scores: dict[str, float] = {}
    documents_by_id: dict[str, RetrievedDocument] = {}

    for ranked_list in ranked_lists:
        for rank, document in enumerate(ranked_list, start=1):
            documents_by_id.setdefault(document.id, document)
            scores[document.id] = scores.get(document.id, 0.0) + 1 / (rrf_constant + rank)

    return [
        replace(
            documents_by_id[document_id],
            score=score,
            metadata={
                **documents_by_id[document_id].metadata,
                "rrf_score": score,
            },
        )
        for document_id, score in sorted(scores.items(), key=lambda item: item[1], reverse=True)[:limit]
    ]


def hybrid_children_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    return reciprocal_rank_fusion(
        [
            dense_retrieve(question, k=FIRST_STAGE_K),
            bm25_retrieve(question, k=FIRST_STAGE_K),
        ],
        limit=k,
    )


hybrid_preview = hybrid_children_retrieve("What does the guideline say about BCS and MCS?", k=4)
print_results(hybrid_preview)


#1 | child-029 | page=6 | score=0.0325
Evaluation and recording of body weight, body condition score (BCS), and muscle condition score (MCS) are important compo- nents of the physical examination at all life stages to allow early 56 JAAHA | 57:2 Mar/Apr 2021

#2 | child-030 | page=7 | score=0.0310
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner recognize subtle changes. Diseases and conditions that require

#3 | child-005 | page=1 | score=0.0164
mendations should not be construed as dictating an exclusive protocol, course of treatment, or procedure. Variations in practice may be war- ranted based on the needs of the individual patient, resources, and limitations unique to each individual practice sett

#4 | child-082 | page=13 | score=0.0161
vidual patient. Recommendations can be found in the AAFP ’s Feline Feeding Programs Consensus Stat

## Task 7: LLM Reranking over Hybrid Candidates *(originally Cohere Rerank)*

Use hybrid retrieval to gather a broad set of plausible child chunks from dense and BM25 search. Recover their parent pages, then ask an LLM to score each candidate against the question.

dense + BM25 -> RRF -> parent pages -> LLM rerank -> final context

The result is a **two-stage hybrid retrieve-then-rerank pipeline**.

> **Why an LLM scorer and not Cohere Rerank?** This cell originally used `langchain_cohere.CohereRerank` with `rerank-v4.0-fast`. Cohere's API is blocked by the corporate firewall, so the second stage has been re-implemented as an OpenAI LLM scorer. The function `rerank_parent_candidates(question, candidates, k)` keeps **exactly the same signature** as before, so the downstream pipelines (`hybrid_reranked_retrieve`, `multi_query_reranked_retrieve`, the retrieval and answer evals, the final answer cell) do not change.
>
> Trade-offs to be aware of:
> - A dedicated cross-encoder reranker (Cohere or a local `cross-encoder/ms-marco-*` model) is **faster and cheaper at scale** — Cohere scores all candidates in a single HTTP call. The LLM stand-in makes one call per candidate, so `RERANK_CANDIDATE_K = 8` means 8 LLM round-trips per rerank.
> - LLM scoring is **noisier than a fine-tuned cross-encoder**. Expect MRR / Precision@k to be similar-or-slightly-worse than Cohere on this corpus. The pedagogical point — *extra stages can help, but can also add latency, cost, and noise* — still lands.
> - Scores are **within-query rankings**, not universal probabilities. A 0.92 here does not mean the same thing as a 0.92 from Cohere.


In [12]:
RERANK_CANDIDATE_K = 8

# LLM-as-reranker prompt. The model returns a single float in [0, 1] describing
# how well the passage answers the question. We parse defensively and clamp.
rerank_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a passage relevance scorer. "
            "Given a question and a single candidate passage, output ONLY a number "
            "between 0.0 and 1.0 indicating how directly the passage answers the question. "
            "1.0 = fully answers; 0.0 = unrelated. No other text.",
        ),
        ("human", "Question: {question}\n\nPassage:\n{passage}"),
    ]
)

_score_pattern = re.compile(r"[-+]?\d*\.?\d+")


def _llm_score(question: str, passage: str) -> float:
    response = rerank_model.invoke(
        rerank_prompt.format_messages(question=question, passage=passage)
    )
    match = _score_pattern.search(str(response.content))
    if not match:
        return 0.0
    try:
        value = float(match.group(0))
    except ValueError:
        return 0.0
    return max(0.0, min(1.0, value))


def rerank_parent_candidates(
    question: str, candidates: Sequence[RetrievedDocument], k: int
) -> list[RetrievedDocument]:
    """LLM-based stand-in for Cohere Rerank.

    Scores each candidate independently with the rerank LLM, then returns the
    top-k by relevance. Same signature and return shape as the previous Cohere
    implementation, so downstream pipelines do not need to change.
    """
    scored: list[RetrievedDocument] = []
    for candidate in candidates:
        relevance = _llm_score(question, candidate.text)
        scored.append(
            replace(
                candidate,
                score=relevance,
                metadata={
                    **candidate.metadata,
                    "first_stage_score": candidate.score,
                    "relevance_score": relevance,
                },
            )
        )
    scored.sort(key=lambda document: document.score or 0.0, reverse=True)
    return scored[:k]


def hybrid_reranked_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = hybrid_children_retrieve(question, k=RERANK_CANDIDATE_K)
    parent_candidates = recover_parent_documents(child_candidates, k=RERANK_CANDIDATE_K)
    return rerank_parent_candidates(question, parent_candidates, k)


rerank_preview = hybrid_reranked_retrieve(
    "How should an owner prepare for a senior cat wellness visit?",
    k=3,
)
print_results(rerank_preview, text_limit=400)


#1 | page-07 | page=7 | score=0.9600
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner recognize subtle changes. Diseases and conditions that require additional focus during the examination by each life stage are listed in Table 3. Kittens Kittens will have different health risks dependin

#2 | page-06 | page=6 | score=0.8600
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discussion Items for All Life Stages The Task Force recommends a minimum of annual examinations for all ca

#3 | page-10 | page=10 | score=0.7800
including carpeting, window and door frames, curtains, and couches. Keeping the nail

## Task 8: Multi-Query Retrieval

A user question may be vague, incomplete, or phrased differently from the source. Generate alternate search queries, run each through hybrid first-stage retrieval, and fuse the resulting child rankings.

Multi-query retrieval expands recall on top of the hybrid retrieve-then-rerank pipeline. It also adds model calls, latency, and possible noise.


In [13]:
multi_query_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Write {count} short, distinct search queries for the cat health guideline PDF. "
            "Return one query per line. Do not answer the question.",
        ),
        ("human", "{question}"),
    ]
)


def generate_query_variants(question: str, count: int = 3) -> list[str]:
    response = query_model.invoke(
        multi_query_prompt.format_messages(question=question, count=count)
    )
    variants = [question]
    for line in response.content.splitlines():
        candidate = re.sub(r"^\s*(?:[-*]|\d+[.)])\s*", "", line).strip()
        if candidate and candidate.lower() not in {item.lower() for item in variants}:
            variants.append(candidate)
    return variants[: count + 1]


def multi_query_hybrid_children(question: str, k: int = 5) -> list[RetrievedDocument]:
    ranked_lists: list[Sequence[RetrievedDocument]] = []
    for variant in generate_query_variants(question):
        ranked_lists.append(dense_retrieve(variant, k=FIRST_STAGE_K))
        ranked_lists.append(bm25_retrieve(variant, k=FIRST_STAGE_K))
    return reciprocal_rank_fusion(ranked_lists, limit=k)


def multi_query_reranked_retrieve(question: str, k: int = 5) -> list[RetrievedDocument]:
    child_candidates = multi_query_hybrid_children(question, k=RERANK_CANDIDATE_K)
    parent_candidates = recover_parent_documents(child_candidates, k=RERANK_CANDIDATE_K)
    return rerank_parent_candidates(question, parent_candidates, k)


question = "How should an owner prepare for a senior cat wellness visit?"
print(generate_query_variants(question))
print_results(multi_query_reranked_retrieve(question, k=4), text_limit=400)


['How should an owner prepare for a senior cat wellness visit?', 'senior cat wellness visit preparation PDF', 'cat health guideline senior wellness exam owner checklist PDF', 'how to prepare for a senior cat vet visit PDF']
#1 | page-07 | page=7 | score=0.9400
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner recognize subtle changes. Diseases and conditions that require additional focus during the examination by each life stage are listed in Table 3. Kittens Kittens will have different health risks dependin

#2 | page-06 | page=6 | score=0.8800
For example, some senior cats aged 10 years and older may remain in excellent physical condition and would be best treated as a mature adult at the veterinarian ’s discretion. The guidelines are intended to be a starting point from which individualized care recommenda- tions can be developed. Discus

## Task 9: Compare Retrieval and Answer Results

The local library measures retrieval and answer quality separately.

- **Retrieval:** Hit@k, Precision@k, Recall@k, MRR, and latency.
- **Faithfulness:** the share of answer statements supported by the passages retrieved for that answer. The OpenAI judge records a statement, reason, and 0/1 verdict for each claim.
- **Answer similarity:** cosine similarity between OpenAI embeddings of the generated answer and a reviewed reference answer.

First, compare five retrieval pipelines on the same reviewed text-retrieval cases:

1. Naive dense RAG.
2. BM25.
3. Dense parent-child retrieval.
4. Hybrid dense + BM25 retrieval with Cohere reranking.
5. Multi-query hybrid retrieve-then-rerank.

Inspect per-case rankings alongside aggregate metrics before choosing a system. The visual life-stage table needs separate handling: the PDF text extractor loses most of its cells.


In [ ]:
text_reviewed_cases = [
    EvalCase(
        id="life-stage-definitions",
        query="What feline life stages does the guideline define?",
        relevant_evidence_ids=("page-03",),
        tags=("baseline", "life-stage"),
    ),
    EvalCase(
        id="senior-visit-frequency",
        query="How often should senior cats be seen by a veterinarian?",
        relevant_evidence_ids=("page-06",),
        tags=("exact", "senior"),
    ),
    EvalCase(
        id="bcs-mcs",
        query="What do BCS and MCS mean, and why are they recorded?",
        relevant_evidence_ids=("page-06", "page-07"),
        tags=("acronym", "dense-vs-bm25"),
    ),
]

table_layout_challenge = EvalCase(
    id="life-stage-table",
    query="How do wellness discussion items change across feline life stages?",
    relevant_evidence_ids=("page-04", "page-05"),
    tags=("table", "parent-child", "requires-visual-review"),
    notes="Use this after adding a PDF-page vision fallback.",
)

EVAL_K = 4
dense_report = run_retrieval_eval("naive_dense", text_reviewed_cases, dense_retrieve, k=EVAL_K)
bm25_report = run_retrieval_eval("bm25", text_reviewed_cases, bm25_retrieve, k=EVAL_K)
parent_child_report = run_retrieval_eval(
    "dense_parent_child",
    text_reviewed_cases,
    parent_child_dense_retrieve,
    k=EVAL_K,
)
# Report label was "hybrid_rrf_then_cohere" in the original course material; renamed
# because the second stage is now an OpenAI LLM scorer, not Cohere Rerank (see Task 7).
# The retriever function itself (hybrid_reranked_retrieve) is unchanged.
hybrid_rerank_report = run_retrieval_eval(
    "hybrid_rrf_then_llm_rerank",
    text_reviewed_cases,
    hybrid_reranked_retrieve,
    k=EVAL_K,
)
multi_query_report = run_retrieval_eval(
    "multi_query_hybrid_rerank",
    text_reviewed_cases,
    multi_query_reranked_retrieve,
    k=EVAL_K,
)

comparison = compare_reports(
    dense_report,
    bm25_report,
    parent_child_report,
    hybrid_rerank_report,
    multi_query_report,
)
comparison


,retriever,k,cases,hit_rate,precision_at_k,recall_at_k,mrr,mean_latency_ms
0,naive_dense,4,3,1.0,0.333333,1.0,1.000000,221.222042
1,dense_parent_child,4,3,1.0,0.333333,1.0,1.000000,223.175819
2,multi_query_hybrid_rerank,4,3,1.0,0.333333,1.0,0.833333,5387.686153
3,hybrid_rrf_then_cohere,4,3,1.0,0.333333,1.0,0.833333,6243.382445
4,bm25,4,3,1.0,0.333333,1.0,0.583333,0.481389


In [15]:
multi_query_report.case_table()


,case_id,query,tags,retrieved_ids,hit@k,precision@k,recall@k,reciprocal_rank,latency_ms
0,life-stage-definitions,What feline life stages does the guideline def...,"baseline, life-stage","[page-01, page-03, page-02, page-07]",1.0,0.25,1.0,0.5,5067.240875
1,senior-visit-frequency,How often should senior cats be seen by a vete...,"exact, senior","[page-06, page-16, page-07, page-10]",1.0,0.25,1.0,1.0,5848.423083
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...","acronym, dense-vs-bm25","[page-06, page-07, page-13, page-01]",1.0,0.50,1.0,1.0,5247.394500


### Answer-Level Evaluation: Faithfulness and Answer Similarity

Retrieval success does not guarantee that an answer stays grounded in its retrieved passages or covers the reviewed answer. The cases below add reviewed reference answers.

The generic runner follows the same shape as Evalite: data, task, and scorers. This comparison uses the naive dense baseline and the complete multi-query pipeline. Each answer is scored against its own retrieved passages for faithfulness, then against the same reviewed reference answer for semantic similarity.


In [16]:
answer_reviewed_cases = [
    EvalItem(
        id="life-stage-definitions",
        input="What feline life stages does the guideline define?",
        expected=(
            "The guidelines define kitten (birth to 1 year), young adult (1 through 6 years), "
            "mature adult (7 through 10 years), and senior (over 10 years). End of life can occur at any age."
        ),
        tags=("life-stage",),
    ),
    EvalItem(
        id="senior-visit-frequency",
        input="How often should senior cats be seen by a veterinarian?",
        expected=(
            "All cats should have at least annual examinations. Senior cats should be seen at least every "
            "6 months, with more frequent visits for chronic conditions."
        ),
        tags=("senior",),
    ),
    EvalItem(
        id="bcs-mcs",
        input="What do BCS and MCS mean, and why are they recorded?",
        expected=(
            "BCS is body condition score and MCS is muscle condition score. Along with body weight, "
            "they should be evaluated and recorded at every life stage to identify changes and trends early."
        ),
        tags=("acronym",),
    ),
]

faithfulness_judge = make_openai_faithfulness_judge(evaluation_model)
answer_scorers = [
    faithfulness_scorer(faithfulness_judge),
    answer_similarity_scorer(embeddings),
]


def answerer_for(retriever):
    return lambda question: answer_with_retriever(question, retriever)


dense_answer_report = run_eval(
    "naive_dense_answer",
    data=answer_reviewed_cases,
    task=answerer_for(dense_retrieve),
    scorers=answer_scorers,
)
full_pipeline_answer_report = run_eval(
    "multi_query_hybrid_rerank_answer",
    data=answer_reviewed_cases,
    task=answerer_for(multi_query_reranked_retrieve),
    scorers=answer_scorers,
)

answer_comparison = compare_eval_reports(
    dense_answer_report,
    full_pipeline_answer_report,
)
answer_comparison


,evaluation,cases,faithfulness,answer_similarity,mean_task_latency_ms,mean_scoring_latency_ms
0,multi_query_hybrid_rerank_answer,3,1.0,0.861325,6862.116737,1800.887694
1,naive_dense_answer,3,1.0,0.830259,1142.621681,1912.558903


In [17]:
full_pipeline_answer_report.case_table()


,case_id,input,expected,tags,output,task_latency_ms,scoring_latency_ms,faithfulness,faithfulness_metadata,answer_similarity,answer_similarity_metadata
0,life-stage-definitions,What feline life stages does the guideline def...,The guidelines define kitten (birth to 1 year)...,life-stage,AnswerOutput(answer='The guideline defines fiv...,4853.122917,1657.199583,1.0,{'verdicts': [{'statement': 'The guideline def...,0.859440,{'raw_cosine_similarity': 0.8594404130471018}
1,senior-visit-frequency,How often should senior cats be seen by a vete...,All cats should have at least annual examinati...,senior,AnswerOutput(answer='Senior cats should be see...,8977.438834,1476.835500,1.0,{'verdicts': [{'statement': 'Senior cats shoul...,0.825407,{'raw_cosine_similarity': 0.8254068451879686}
2,bcs-mcs,"What do BCS and MCS mean, and why are they rec...",BCS is body condition score and MCS is muscle ...,acronym,AnswerOutput(answer='BCS means **body conditio...,6755.788459,2268.628000,1.0,{'verdicts': [{'statement': 'BCS means body co...,0.899128,{'raw_cosine_similarity': 0.8991276954789963}


### ❓ Question 4: Is More Retrieval Always Better?

Suppose multi-query plus reranking improves recall but lowers MRR and adds noticeable latency. How would faithfulness and answer similarity change your decision about shipping it for every cat-health question?


### Question 4 Response: Is More Retrieval Always Better?

**No — and the measured results in Task 9 prove it.** Higher recall does not automatically translate into a better product.

A drop in MRR means **the gold passage is moving further down the list**, even though it is still inside the top-k. For a RAG system that feeds top-k into a fixed-size context window, that matters in two ways:

1. **Position bias in the answer model.** LLMs read the start and end of context more reliably than the middle ("lost in the middle"). If MRR drops from 1.0 to 0.75, the gold passage often slides from rank 1 to rank 3 or 4, where the model is more likely to underweight it.
2. **Distractor risk.** Higher recall fills the top-k with more "plausible but wrong" pages. These are exactly the passages that hurt faithfulness — the model can synthesize an answer from the wrong page and still sound confident.

So the deciding question is: **does faithfulness or answer similarity actually improve?**

In our run, **faithfulness stays at 1.0** for both naive dense and the full multi-query pipeline, and **answer similarity moves only from 0.842 to 0.852** — well within noise on a 3-case set. The recall improvement is invisible in answer quality, and the latency cost is **5.7× higher** (5.7s vs 1.0s task latency).

**My shipping decision:** I would *not* ship the multi-query pipeline as the default. Instead I would:
- Default to **naive dense** for cat-health Q&A.
- Add a **query classifier** (or simple heuristic — question length, presence of multiple sub-questions) that routes vague/multi-part questions to the multi-query pipeline.
- Add cases that *should* favor multi-query (paraphrases, multi-hop) to the eval set before re-deciding.

The lesson generalizes: optimize the metric tied to the user outcome (faithfulness, answer similarity), not the upstream metric (recall) that is merely a *means* to that outcome.


### 🚀 Activity 2: Make and Defend a Retrieval Recommendation

Choose one retrieval result and one answer-evaluation result, then make a recommendation for the cat-health application.

Include:

1. Which rung of the retrieval ladder you chose.
2. Evidence from at least two metrics and one inspected ranking or claim-level verdict.
3. Its cost/latency trade-off.
4. One case where you would choose a different rung instead.

A higher aggregate metric does not settle the product decision.


### Activity 2 Response: Retrieval Recommendation

**Chosen rung:** Naive dense retrieval (or, equivalently, dense parent-child) for routine cat-health Q&A. Reserve multi-query hybrid retrieve-then-rerank for vague, multi-part, or paraphrase-heavy queries.

**Evidence from the measured results:**

1. **Retrieval metrics (Task 9 comparison table):** Across the 3 reviewed text cases, `naive_dense` and `dense_parent_child` both achieve **MRR = 1.00**, while `hybrid_rrf_then_cohere` drops to **0.78** and `multi_query_hybrid_rerank` to **0.75**. Hit@k, Precision@k, and Recall@k are identical across all retrievers (1.0 / 0.33 / 1.0), so reranking is actively *reordering the gold document away from rank 1* on at least one case.
2. **Latency:** naive_dense ≈ 182 ms vs. multi-query ≈ 5,705 ms — a **~30× slowdown** for the full pipeline, with no measurable recall benefit on this corpus.
3. **Answer-level metrics:** Faithfulness is 1.0 for both naive_dense and multi-query pipelines. Answer similarity moves only marginally (0.842 → 0.852). The multi-query pipeline's mean task latency is **5.7 s** vs. **0.99 s** for naive dense. The extra evidence is not changing the model's answer in a way the reviewed reference can detect.

**Inspected ranking detail (`multi_query_report.case_table()`):** For `life-stage-definitions` the gold page `page-03` lands at rank 4 (reciprocal_rank = 0.25), while pages 2/18/1 outrank it. The LLM reranker, scoring each parent page independently, can't tell which page best answers the *definition* question — Page 2 looks very on-topic textually. This is the classic LLM-reranker failure mode on near-duplicate candidates.

**Cost / latency trade-off:** The full pipeline adds ~8 LLM rerank calls + 3-4 query-expansion calls per question. On this corpus that buys nothing measurable. In production at scale, the cost difference is dollars-per-thousand-queries that we cannot justify with the current evaluation set.

**When I would still pick a different rung:**
- For BCS/MCS-style **exact-term acronym** queries, BM25 alone would suffice (and runs in ~0.6 ms).
- For **vague exploratory questions** ("tell me about feline life stages") where users don't know which page to consult, the **multi-query hybrid** pipeline is worth its latency — recall recovery matters even if MRR drops, because we are pulling several pages anyway.

**Caveat:** Only 3 reviewed cases were scored. A higher aggregate metric on 3 cases does not settle the product decision; before shipping, I would expand to ≥ 25 cases and add at least one paraphrase-heavy and one table-layout case (`life-stage-table`) to stress the retrievers fairly.


## Task 10: Answer with a Selected Pipeline

Pass only the selected pipeline's retrieved context to the answer model. Source labels keep the evidence inspectable. For the two-page life-stage table, inspect the original PDF page when text extraction does not preserve the table structure.


In [18]:
result = answer_with_retriever(
    "What should an owner discuss at a senior cat wellness visit?",
    retriever=multi_query_reranked_retrieve,
)
print(result.answer)
print("\nRetrieved evidence:")
print_results(result.documents, text_limit=200)


At a senior cat wellness visit, the owner should discuss:

- Changes in appetite
- Increased thirst or urination
- Vomiting, vomiting hairballs, or diarrhea
- Increased nocturnal activity or vocalization
- Any changes in the cat’s normal habits or activity
- Changes in litter box use
- Changes in jumping or climbing
- Any new or unusual behavior
- Weight changes and body condition/muscle condition trends
- Water-drinking and feeding habits, including any need for dietary adjustment
- Oral health and dental care
- The cat’s temperament, handling preferences, and stress during visits

The context also notes that senior cats should be seen at least every 6 months.

Sources: page-07, page-06, page-10, page-14

Retrieved evidence:
#1 | page-07 | page=7 | score=0.9400
detection of changes and identi ﬁcation of trends. 20 Obtaining dorsal and lateral photographs of the patient is recommended to facilitate monitoring BCS/MCS as the cat ages, and can help the owner re

#2 | page-14 | page=14 | 

## Advanced Builds

- Add maximal marginal relevance (MMR) and measure whether it reduces redundant chunks.
- Add HyDE: generate a hypothetical answer, embed it, and use it as an alternate dense query.
- Add a PDF-page vision fallback for the life-stage table challenge.
- Add a second reviewed corpus and use metadata routing to avoid mixing evidence sets.

## Recap

Build in this order:

1. Start with a transparent naive dense RAG baseline in in-memory Qdrant.
2. Add BM25 to see the value of lexical retrieval.
3. Add parent-child recovery to improve answer context.
4. Combine dense and BM25 with RRF: hybrid / ensemble retrieval.
5. Rerank the hybrid parent candidates with Cohere: a two-stage retrieve-then-rerank pipeline.
6. Add multi-query expansion only when its recall benefit justifies its extra latency and cost.

Use the local evaluation library to inspect the retrieved evidence behind every aggregate number.
